# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [24]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

## Load the document- Peter Drucker

In [25]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf")
docs = loader.load()

# Join pages
document_text = "\n".join([page.page_content for page in docs])
print(document_text[:500])  # preview


www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


## Define Structured outpue model

In [26]:
from pydantic import BaseModel

class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


## Initialize the Client

In [27]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


## Developer and User prompts

In [28]:
developer_prompt = (
    "You are an expert summarizer who produces structured JSON outputs "
    "according to the Pydantic model provided. "
    "Ensure the tone of the summary is in 'Formal Academic Writing' style."
)

user_prompt = f"""
Summarize the article below and output a JSON object that fits the SummaryOutput model:
Fields: Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens.

Article content:
{document_text[:4000]}  # shortened for token limit
"""


## Structured Summary

In [29]:
response = client.chat.completions.create(
    model="gpt-4o-mini",   # non-GPT-5 model per assignment instructions
    messages=[
        {"role": "system", "content": developer_prompt},
        {"role": "user", "content": user_prompt}
    ],
    response_format={"type": "json_object"},
)

# Parse the JSON output
try:
    output = SummaryOutput.model_validate_json(response.choices[0].message.content)
    output.InputTokens = response.usage.prompt_tokens
    output.OutputTokens = response.usage.completion_tokens
    print(output)
except ValidationError as e:
    print("Validation error:", e)


Author='Peter F. Drucker' Title='Managing Oneself' Relevance='This article discusses the importance of self-knowledge in achieving professional success in the knowledge economy, encouraging individuals to take responsibility for their career paths.' Summary="In the contemporary professional landscape, success is largely contingent upon an individual's self-understanding, encompassing their strengths, weaknesses, work styles, and values. Drucker posits that modern employees must assume the role of their own chief executive officer. He emphasizes that recognizing one's strengths through systematic feedback analysis is crucial, as is understanding one's optimal working conditions and aligning personal values with organizational ethics. Ultimately, identifying the right work environment enhances contributions and fosters excellence in performance." Tone='Formal Academic Writing' InputTokens=1015 OutputTokens=176


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

## Evaluate the summary

In [30]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a concise academic summarizer. "
                       "Always output the result strictly as a valid JSON object "
                       "that matches the required schema."
        },
        {
            "role": "user",
            "content": "Summarize 'Managing Oneself' by Peter Drucker and return the result as JSON."
        },
    ],
    response_format={"type": "json_object"},
)


In [31]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a concise academic summarizer. "
                       "Always return the output strictly as a valid JSON object "
                       "that conforms to the required fields."
        },
        {
            "role": "user",
            "content": "Summarize 'Managing Oneself' by Peter Drucker. Output the result as JSON."
        }
    ],
    response_format={"type": "json_object"},
)

print(response.choices[0].message.content)


{
  "title": "Managing Oneself",
  "author": "Peter Drucker",
  "summary": {
    "key_concepts": [
      "Self-awareness: Understand one's strengths, weaknesses, and values.",
      "Responsibility for one's own development: Individuals must take charge of their own learning and effectiveness.",
      "Feedback analysis: Regularly review and assess personal strengths and contributions to improve performance.",
      "Building on strengths: Focus on enhancing and leveraging one's strengths rather than trying to fix weaknesses.",
      "Personal missions: Establish a clear personal vision and set of goals to guide decisions and actions."
    ],
    "main_takeaways": [
      "Effective self-management leads to greater professional success and personal satisfaction.",
      "Understanding how one performs and learns can lead to better career choices and opportunities.",
      "Maintaining a commitment to continuous learning and adaptation is essential in a rapidly changing world."
    ]
  

In [32]:
from pydantic import BaseModel

class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int


In [34]:
data = {
  "Author": "Peter Drucker",
  "Title": "Managing Oneself",
  "Relevance": "The article provides timeless insights into self-management, essential for professionals navigating rapid change and AI-driven workplaces.",
  "Summary": "To thrive in a knowledge economy, Drucker argues individuals must act as their own CEOs—identifying strengths, values, and optimal environments while taking responsibility for lifelong learning and collaboration.",
  "Tone": "Formal Academic Writing",
  "InputTokens": response.usage.prompt_tokens,
  "OutputTokens": response.usage.completion_tokens
}

output = SummaryOutput(**data)
print(output)


Author='Peter Drucker' Title='Managing Oneself' Relevance='The article provides timeless insights into self-management, essential for professionals navigating rapid change and AI-driven workplaces.' Summary='To thrive in a knowledge economy, Drucker argues individuals must act as their own CEOs—identifying strengths, values, and optimal environments while taking responsibility for lifelong learning and collaboration.' Tone='Formal Academic Writing' InputTokens=55 OutputTokens=215


## Deep Evaluation

In [35]:
import deepeval
print(deepeval.__version__)


3.6.8


In [36]:
import deepeval
print(deepeval.__version__)


3.6.8


In [37]:
import deepeval
print(deepeval.__version__)

3.6.8


## Deep evaluation


In [38]:
import deepeval, pkgutil
import deepeval.metrics as m

print("DeepEval version:", deepeval.__version__)
print("Available metric submodules:")
for mod in pkgutil.iter_modules(m.__path__):
    print("  ", mod.name)


DeepEval version: 3.6.8
Available metric submodules:
   answer_relevancy
   api
   arena_g_eval
   argument_correctness
   base_metric
   bias
   contextual_precision
   contextual_recall
   contextual_relevancy
   conversation_completeness
   conversational_dag
   conversational_g_eval
   dag
   faithfulness
   g_eval
   goal_accuracy
   hallucination
   indicator
   json_correctness
   knowledge_retention
   mcp
   mcp_use_metric
   misuse
   multimodal_metrics
   non_advice
   pii_leakage
   plan_adherence
   plan_quality
   prompt_alignment
   ragas
   role_adherence
   role_violation
   step_efficiency
   summarization
   task_completion
   tool_correctness
   tool_use
   topic_adherence
   toxicity
   turn_relevancy
   utils


In [39]:
import deepeval, inspect, pkgutil, importlib

for mod in pkgutil.iter_modules(deepeval.metrics.__path__):
    if "summar" in mod.name:
        print("Possible module:", mod.name)


Possible module: summarization


In [40]:
import importlib
mod = importlib.import_module("deepeval.metrics.summarization")
print(dir(mod))


['SummarizationTemplate', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'schema', 'summarization', 'template']


In [41]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric


In [42]:
evaluation_questions = [
    "Does the summary accurately capture Drucker’s core ideas about managing oneself?",
    "Are strengths, feedback analysis, and personal responsibility represented?",
    "Is the summary concise but complete?",
    "Does it flow logically from idea to idea?",
    "Is the tone formal and academic?",
    "Does it avoid emotional or promotional phrasing?",
    "Is the content factual, respectful, and safe to share professionally?"
]


In [43]:
from langchain_community.document_loaders import PyPDFLoader

# Load Drucker article
loader = PyPDFLoader("https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf")
docs = loader.load()

# Join the pages into a single string
document_text = "\n".join([page.page_content for page in docs])

print(len(document_text), "characters loaded")


51451 characters loaded


In [44]:
summary_text = (
    "Peter F. Drucker argues that success in the knowledge economy depends "
    "on managing oneself—understanding one’s strengths, values, and preferred "
    "work environment. He recommends systematic feedback analysis, aligning "
    "personal and organizational values, and taking responsibility for one’s "
    "development and contributions."
)


In [47]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric

# assume `document_text` = your loaded Drucker article text
# and `summary_text` = your summarized version
# if document_text is long, slicing keeps it under token limits

test_case = LLMTestCase(
    input=document_text[:4000],          # context given to the model
    expected_output=document_text[:1000],# partial ground truth reference
    actual_output=summary_text,          # model’s generated summary
)

summarization_metric = SummarizationMetric(model="gpt-4o-mini")

# run the evaluation
evaluate(
    test_cases=[test_case],
    metrics=[summarization_metric],
)

# display the results
print({
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
})


✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.6, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.60 because the summary includes extra information about Drucker and his recommendations that were not present in the original text, which may mislead the reader. Additionally, it fails to answer a specific question regarding the authorship of the article, indicating a lack of completeness., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they 

⚠ WARNING: No hyperparameters logged.
» ]8;id=179855;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 9.18s | token cost: 0.0011269499999999998 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{'SummarizationScore': None, 'SummarizationReason': None}


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [2]:
%load_ext dotenv
%dotenv ../05_src/.secrets

In [4]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

try:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Say hello in one short sentence."}]
    )
    print("✅ API key works. Response:")
    print(response.choices[0].message.content)
except Exception as e:
    print("❌ Something went wrong:", e)


✅ API key works. Response:
Hello!


In [5]:
import os

print("OpenAI Key Present:", bool(os.getenv("OPENAI_API_KEY")))
print("Has document_text:", 'document_text' in locals())
print("Has summary_text:", 'summary_text' in locals())


OpenAI Key Present: True
Has document_text: False
Has summary_text: False


In [6]:
# --- Load Drucker article into document_text ---
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf")
docs = loader.load()

# Join all pages into a single string
document_text = "\n".join(page.page_content for page in docs)

print("✅ document_text loaded, length:", len(document_text))
print(document_text[:500])  # preview first 500 chars


✅ document_text loaded, length: 51451
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


In [7]:
# --- Define the original summary ---
summary_text = (
    "Peter F. Drucker argues that success in the knowledge economy depends "
    "on managing oneself—understanding one’s strengths, values, and preferred "
    "work environment. He recommends systematic feedback analysis, aligning "
    "personal and organizational values, and taking responsibility for one’s "
    "development and contributions."
)
print("✅ summary_text defined, length:", len(summary_text))


✅ summary_text defined, length: 316


In [8]:
import os
print("OpenAI Key Present:", bool(os.getenv("OPENAI_API_KEY")))
print("Has document_text:", 'document_text' in locals())
print("Has summary_text:", 'summary_text' in locals())


OpenAI Key Present: True
Has document_text: True
Has summary_text: True


In [9]:
# Load Drucker article
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf")
docs = loader.load()
document_text = "\n".join(p.page_content for p in docs)

# Restore summary
summary_text = (
    "Peter F. Drucker argues that success in the knowledge economy depends "
    "on managing oneself—understanding one’s strengths, values, and preferred "
    "work environment. He recommends systematic feedback analysis, aligning "
    "personal and organizational values, and taking responsibility for one’s "
    "development and contributions."
)


In [10]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric

# Define your baseline test
test_case = LLMTestCase(
    input=document_text[:4000],
    expected_output=document_text[:1000],
    actual_output=summary_text,
)

summarization_metric = SummarizationMetric(model="gpt-4o-mini")

evaluate(
    test_cases=[test_case],
    metrics=[summarization_metric],
)

base_score = summarization_metric.score
base_reason = summarization_metric.reason

print({"BaseScore": base_score, "BaseReason": base_reason})


✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.6, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.60 because the summary contains a significant contradiction regarding the concept of self-management versus self-knowledge, which misrepresents the original text's message. Additionally, it introduces extra information about aligning values that is not present in the original text, leading to potential confusion. However, it does not leave out critical details that would hinder understanding., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in 

⚠ WARNING: No hyperparameters logged.
» ]8;id=36740;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 12.96s | token cost: 0.0012220500000000001 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{'BaseScore': None, 'BaseReason': None}


In [11]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

improve_prompt = f"""
You are a summarization editor.

Source excerpt:
{document_text[:2000]}

Previous summary:
{summary_text}

Feedback:
"The summary includes extra information not present in the source and misses authorship details."

Task:
Revise the summary to stay faithful to the text, mention Peter F. Drucker explicitly,
and keep it under 120 words in a formal academic tone.
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": improve_prompt}],
)
enhanced_summary = response.choices[0].message.content
print("Enhanced Summary:\n", enhanced_summary)


Enhanced Summary:
 In "Managing Oneself," Peter F. Drucker emphasizes that success in the knowledge economy requires individuals to understand themselves, including their strengths, weaknesses, values, and optimal work environments. He asserts that in today's landscape, where companies do not manage careers, individuals must act as their own chief executive officers. To thrive, one must engage in self-reflection and cultivate self-knowledge to navigate a lengthy work life effectively. Drucker underscores the importance of self-awareness in ensuring productivity and making informed career decisions. Overall, successful management of one's career depends on a deep understanding of personal attributes and contributions.


In [15]:
# --- Evaluate the enhanced summary ---
evaluate(
    test_cases=[test_case_enhanced],
    metrics=[summarization_metric_enhanced],
)

# Retrieve results directly from the metric instance
enhanced_score = summarization_metric_enhanced.score
enhanced_reason = summarization_metric_enhanced.reason

print({
    "Before": base_score,
    "After": enhanced_score,
    "Improvement": (enhanced_score - base_score) if base_score and enhanced_score else "N/A",
    "EnhancedReason": enhanced_reason,
})


✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 1.00 because the summary accurately reflects the original text without any contradictions or extra information, demonstrating a perfect alignment with the source material., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Co

⚠ WARNING: No hyperparameters logged.
» ]8;id=764237;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 8.26s | token cost: 0.001191 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{'Before': None, 'After': None, 'Improvement': 'N/A', 'EnhancedReason': None}


In [16]:
summarization_metric_enhanced = SummarizationMetric(model="gpt-4o-mini")

evaluate(
    test_cases=[test_case_enhanced],
    metrics=[summarization_metric_enhanced],
)

# DeepEval 3.6.x usually logs results directly


✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 1.00 because the summary accurately reflects the original text without any contradictions or extra information, demonstrating a perfect alignment., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an in

⚠ WARNING: No hyperparameters logged.
» ]8;id=11637;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 15.18s | token cost: 0.0011870999999999997 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Summarization', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the summary accurately reflects the original text without any contradictions or extra information, demonstrating a perfect alignment.', strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.0011870999999999997, verbose_logs='Truths (limit=None):\n[\n    "Success in the knowledge economy comes to those who know themselves.",\n    "Companies today aren’t managing their knowledge workers’ careers.",\n    "Individuals must be their own chief executive officer.",\n    "It is important to keep oneself engaged and productive during a work life that may span some 50 years.",\n    "Understanding one\'s strengths and weaknesses is crucial for career success.",\n    "Feedback analysis can help identify strengths and areas for improvement.",\n    "Individuals should concentrate

In [17]:
import io
import sys

buffer = io.StringIO()
sys.stdout = buffer

evaluate(
    test_cases=[test_case_enhanced],
    metrics=[summarization_metric_enhanced],
)

sys.stdout = sys.__stdout__
output_text = buffer.getvalue()

print(output_text)  # show the captured output


✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

⚠ WARNING: No hyperparameters logged.
» ]8;id=828913;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 20.67s | token cost: 0.0011979 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [18]:
import re
match = re.search(r"score:\s*([\d.]+)", output_text)
enhanced_score = float(match.group(1)) if match else None
print("Enhanced Score:", enhanced_score)


In [19]:
print({
    "Before": base_score,
    "After": enhanced_score,
    "Improvement": enhanced_score - base_score if base_score and enhanced_score else "N/A"
})


In [20]:
import io, sys, re
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase

# Re-create the metric
summarization_metric_enhanced = SummarizationMetric(model="gpt-4o-mini")

# Capture printed DeepEval output
buf = io.StringIO()
sys_stdout = sys.stdout
sys.stdout = buf

evaluate(
    test_cases=[test_case_enhanced],
    metrics=[summarization_metric_enhanced],
)

# restore normal printing
sys.stdout = sys_stdout

# Get the raw printed text
eval_output = buf.getvalue()
print("Captured evaluation output:\n")
print(eval_output)


✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

⚠ WARNING: No hyperparameters logged.
» ]8;id=263011;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 10.15s | token cost: 0.00116325 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [21]:
import io, sys, re
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase

# Re-create the metric
summarization_metric_enhanced = SummarizationMetric(model="gpt-4o-mini")

# Capture printed DeepEval output
buf = io.StringIO()
sys_stdout = sys.stdout
sys.stdout = buf

evaluate(
    test_cases=[test_case_enhanced],
    metrics=[summarization_metric_enhanced],
)

# restore normal printing
sys.stdout = sys_stdout

# Get the raw printed text
eval_output = buf.getvalue()
print("Captured evaluation output:\n")
print(eval_output)


✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

⚠ WARNING: No hyperparameters logged.
» ]8;id=548545;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 13.03s | token cost: 0.00133155 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Please, do not forget to add your comments.

## The enhanced summary achieved a full passing score (100% pass rate), compared to the baseline score of 0.6. The revised version stayed faithful to Peter F. Drucker’s original ideas, mentioned the author explicitly, and removed extraneous details. This demonstrates that structured evaluation feedback can effectively guide iterative refinement in summarization tasks.

In [23]:
import pandas as pd
from IPython.display import display, HTML

# Wrap your summaries into a DataFrame for a clean side-by-side layout
comparison_df = pd.DataFrame({
    "Version": ["Original Summary", "Enhanced Summary"],
    "Summary Text": [summary_text, enhanced_summary]
})

# Add a small header and display neatly
display(HTML("<h3>🧩 Summary Comparison</h3>"))
display(comparison_df.style.set_table_styles([
    {"selector": "th", "props": [("background-color", "#f5f5f5"), ("font-weight", "bold")]},
    {"selector": "td", "props": [("vertical-align", "top"), ("white-space", "pre-wrap")]}
]).hide(axis="index"))


Version,Summary Text
Original Summary,"Peter F. Drucker argues that success in the knowledge economy depends on managing oneself—understanding one’s strengths, values, and preferred work environment. He recommends systematic feedback analysis, aligning personal and organizational values, and taking responsibility for one’s development and contributions."
Enhanced Summary,"In ""Managing Oneself,"" Peter F. Drucker emphasizes that success in the knowledge economy requires individuals to understand themselves, including their strengths, weaknesses, values, and optimal work environments. He asserts that in today's landscape, where companies do not manage careers, individuals must act as their own chief executive officers. To thrive, one must engage in self-reflection and cultivate self-knowledge to navigate a lengthy work life effectively. Drucker underscores the importance of self-awareness in ensuring productivity and making informed career decisions. Overall, successful management of one's career depends on a deep understanding of personal attributes and contributions."



# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
